# Урок 15 · Медленная версия: слои свёрточной сети

На прошлом ноутбуке мы подготовили данные. Теперь разбираем **сердце CNN** — три слоя:
`Conv2D` → `MaxPooling2D` → `Flatten`. По одному, медленно, с картинками.

> 💡 Правило урока: сначала смотрим, ЧТО делает один слой с картинкой глазами, и только потом собираем их вместе.

## Шаг 0. Готовим данные (как в прошлый раз)

Клетка самодостаточна — запускается с чистого листа.

In [ ]:
from tensorflow.keras.datasets import cifar10
import numpy as np, matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = X_train / 255.0   # нормализация: 0-255 -> 0-1
X_test  = X_test  / 255.0

classes = ["самолёт","машина","птица","кошка","олень",
           "собака","лягушка","лошадь","корабль","грузовик"]
print("Готово. Форма данных:", X_train.shape)

## Шаг 1. Что делает ОДИН слой Conv2D

`Conv2D` — это набор фильтров-рамок, которые скользят по картинке (мы видели это в демо).
Проверим на одной картинке, что слой выдаёт **карты признаков**.

In [ ]:
from tensorflow.keras.layers import Conv2D
import tensorflow as tf

# берём одну картинку и добавляем "фиктивную" ось батча:
# слои Keras ждут пачку картинок, даже если картинка одна -> форма (1, 32, 32, 3)
one = X_train[0:1]
print("Форма на входе:", one.shape)

# создаём слой: 8 фильтров, каждый размером 3x3
conv = Conv2D(filters=8, kernel_size=(3, 3), activation="relu")

# пропускаем картинку через слой
feature_maps = conv(one)
print("Форма на выходе:", feature_maps.shape)
# было 3 канала цвета -> стало 8 карт признаков (по одной на фильтр)

**❓ Вопрос 1.** На входе было `(1, 32, 32, 3)`, на выходе `(1, 30, 30, 8)`.

- Откуда взялось `8`? → ...
- Почему `32` стало `30`? → ...

<details><summary>Подсказка</summary>

8 = столько фильтров мы задали. 30 = рамка 3×3 не может встать вплотную к краю,
поэтому по краям теряется по 1 пикселю с каждой стороны (32−2 = 30).
</details>

In [ ]:
# посмотрим на 4 карты признаков глазами
plt.figure(figsize=(8,2))
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap="gray")  # i-я карта признаков
    plt.title("фильтр " + str(i))
    plt.axis("off")
plt.show()
# каждый фильтр подсветил СВОИ детали картинки

## Шаг 2. Что делает MaxPooling2D

`MaxPooling2D` **уменьшает** карту, оставляя из каждого квадрата 2×2 только самое большое число.
Зачем: сеть становится быстрее и меньше цепляется за мелкие сдвиги.

In [ ]:
from tensorflow.keras.layers import MaxPooling2D

pool = MaxPooling2D(pool_size=(2, 2))
pooled = pool(feature_maps)

print("До пулинга:   ", feature_maps.shape)
print("После пулинга:", pooled.shape)
# ширина и высота уменьшились примерно вдвое

**❓ Вопрос 2.** Было `(1, 30, 30, 8)`, стало `(1, 15, 15, 8)`.
Число карт (8) изменилось? А их размер? Почему пулинг НЕ трогает число фильтров?

Впиши код, чтобы увидеть уменьшение глазами (замени `...`):

In [ ]:
plt.figure(figsize=(6,3))
plt.subplot(1,2,1); plt.imshow(feature_maps[0,:,:,0], cmap="gray"); plt.title("до 30x30"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(pooled[0,:,:,...], cmap="gray"); plt.title("после 15x15"); plt.axis("off")  # ← допиши: 0
plt.show()

## Шаг 3. Что делает Flatten

Свёрточные слои работают с картинками (2D). А финальный `Dense`-слой, который выдаёт ответ,
понимает только **длинный ряд чисел** (1D). `Flatten` вытягивает картинку в один ряд.

In [ ]:
from tensorflow.keras.layers import Flatten

flat = Flatten()(pooled)
print("До Flatten:", pooled.shape)      # (1, 15, 15, 8) - объёмная
print("После Flatten:", flat.shape)     # (1, 1800)      - один ряд
print("Проверим: 15 * 15 * 8 =", 15*15*8)  # ровно столько чисел

**❓ Вопрос 3.** Почему `Flatten` обязателен ПЕРЕД `Dense`, но НЕ нужен перед `Conv2D`?

<details><summary>Подсказка</summary>

Conv2D работает с картинкой и учитывает, ГДЕ находится пиксель. Dense видит только плоский список
и в пространстве не разбирается — поэтому картинку сначала распрямляют.
</details>

## Шаг 4. Собираем всё вместе

Теперь те же три слоя — но как настоящую модель. Смотри: это ровно то, что мы разбирали по частям.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(32,32,3)),  # шаг 1
    MaxPooling2D((2,2)),                                          # шаг 2
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D((2,2)),
    Flatten(),                                                    # шаг 3
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),   # 10 категорий -> 10 вероятностей
])

# summary() показывает форму после КАЖДОГО слоя - сверь с тем, что мы считали руками
model.summary()

**❓ Вопрос 4.** Найди в таблице `model.summary()` строку с `flatten`.
Какое число там в форме выхода? Сходится ли оно с логикой «высота × ширина × число фильтров»?

---
## 🎯 Задания

### 🟢 Базовый
Обучи собранную модель на 3 эпохи и запиши точность.

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=3, validation_data=(X_test, y_test))

### 🟡 Продвинутый
Поменяй в ПЕРВОМ слое число фильтров с 32 на 16. Как изменилась строка `flatten` в `summary()`
и точность? Запиши обе цифры до и после.

### ⭐ Со звёздочкой
Собери модель БЕЗ свёрточных слоёв (сразу `Flatten` → `Dense` → `Dense`) на тех же данных.
Обучи 3 эпохи и сравни точность с CNN. Насколько CNN выигрывает и почему?

In [ ]:
# твой код здесь: Sequential([Flatten(input_shape=(32,32,3)), Dense(...), Dense(10, activation="softmax")])


---
## Мини-итог

Заполни своими словами:

- `Conv2D` делает ...
- `MaxPooling2D` делает ...
- `Flatten` нужен, потому что ...

> Теперь `model.summary()` для тебя не таблица-загадка, а понятная цепочка: картинка → карты признаков → уменьшение → ряд чисел → ответ.